In [40]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'


In [41]:
from models import Ride, ride_deserializer

In [42]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id=None,
    value_deserializer=ride_deserializer
)

/tmp/ipykernel_3538/3234239239.py:1: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


In [43]:
from datetime import datetime

print(f"Listening to {topic_name}...")

messages = consumer.poll(timeout_ms=5000, max_records=10)
count = 0
for records in messages.values():
    for message in records:
        ride = message.value
        pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
        print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
              f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
              f"pickup={pickup_dt}")
        count += 1
        if count >= 10:
            break
    if count >= 10:
        break

if count == 0:
    print("No rides received within 5 seconds.")

consumer.close()

Listening to rides...
Received: PU=1, DO=10, distance=10, amount=$20.00, pickup=2025-11-01 00:13:25
Received: PU=43, DO=186, distance=1.68, amount=$22.15, pickup=2025-11-01 00:13:25
Received: PU=142, DO=237, distance=2.28, amount=$24.94, pickup=2025-11-01 00:49:07
Received: PU=163, DO=238, distance=2.7, amount=$25.62, pickup=2025-11-01 00:07:19
Received: PU=138, DO=261, distance=12.87, amount=$86.14, pickup=2025-11-01 00:00:00
Received: PU=138, DO=37, distance=8.4, amount=$48.65, pickup=2025-11-01 00:18:50
Received: PU=90, DO=100, distance=0.85, amount=$16.45, pickup=2025-11-01 00:21:11
Received: PU=142, DO=170, distance=3.01, amount=$25.85, pickup=2025-11-01 00:07:31
Received: PU=237, DO=144, distance=3.82, amount=$57.54, pickup=2025-11-01 00:46:52
Received: PU=162, DO=161, distance=0.89, amount=$12.95, pickup=2025-11-01 00:56:59


In [44]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-to-postgres',
    value_deserializer=ride_deserializer
)

/tmp/ipykernel_3538/1377099248.py:6: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


In [45]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...
Inserted 2100 rows...
Inserted 2200 rows...
Inserted 2300 rows...
Inserted 2400 rows...
Inserted 2500 rows...
Inserted 2600 rows...
Inserted 2700 rows...
Inserted 2800 rows...
Inserted 2900 rows...
Inserted 3000 rows...
Inserted 3100 rows...
Inserted 3200 rows...
Inserted 3300 rows...
Inserted 3400 rows...
Inserted 3500 rows...
Inserted 3600 rows...
Inserted 3700 rows...
Inserted 3800 rows...
Inserted 3900 rows...
Inserted 4000 rows...
